In [9]:
# OpenAI Vision API configuration
VISION_MODEL = "gpt-4.1-mini"
MAX_IMAGE_SIZE_MB = 25
MAX_RETRIES = 3

# Cache configuration
CACHE_TTL_SECONDS = 3600  # 1 hour

# Receipt JSON structure template
RECEIPT_JSON_TEMPLATE = {
    "merchant": "",
    "date": "",
    "total": 0.0,
    "items": []
}

In [ ]:
import base64
import os
from pathlib import Path
from typing import Optional
from PIL import Image
from config import MAX_IMAGE_SIZE_MB


class ImageProcessor:
    """Processes images for Vision API consumption."""

    def __init__(self):
        """Initialize the ImageProcessor."""
        pass

    def encode_image(self, image_path: str) -> str:
        """
        Encode image to base64 with proper data URI prefix.
        """
        # Determine image format
        path = Path(image_path)
        ext = path.suffix.lower()

        if ext != ".jpeg":
            image_path = self.convert_to_jpeg(image_path)
            ext = ".jpeg"

        with open(image_path, "rb") as f:
            image_data = f.read()

        base64_string = base64.b64encode(image_data).decode("utf-8")

        return f"data:image/{ext.replace('.', '')};base64,{base64_string}"

    def validate_image_size(self, image_path: str) -> bool:
        """
        Validate that image file size does not exceed API limit.
        """
        file_size_bytes = os.path.getsize(image_path)
        file_size_mb = file_size_bytes / (1024 * 1024)

        if file_size_mb > MAX_IMAGE_SIZE_MB:
            raise ValueError(f"Image size exceeds maximum: {MAX_IMAGE_SIZE_MB}MB limit.")

        return True

    def convert_to_jpeg(
        self, image_path: str, output_path: Optional[str] = None
    ) -> str:
        """
        Convert PNG image to JPEG format.
        """
        if output_path is None:
            output_path = str(Path(image_path).with_suffix(".jpeg"))

        with Image.open(image_path) as img:
            rgb_image = img.convert("RGB")
            rgb_image.save(output_path, format="JPEG")

        return output_path


In [ ]:
import json
from typing import Dict, Any, Optional
from api_client import get_openai_client
from config import VISION_MODEL, RECEIPT_JSON_TEMPLATE


class ReceiptExtractor:
    """Extracts structured data from receipt images."""

    def __init__(self):
        """Initialize the ReceiptExtractor."""
        self.client = get_openai_client()

    def extract_receipt_data(self, image_base64: str) -> Dict[str, Any]:
        """
        Extract structured receipt data from image.
        """
        prompt = """
            Extract receipt information from this image and return it as JSON.

            The JSON structure should include the following fields:
            - merchant: The name of the merchant/vendor
            - date: The date of the purchase
            - total: The total amount spent
            - items: A list of items purchased, each with a name and price

            Example output:
            {
                "merchant": "Example Store",
                "date": "2023-01-01",
                "total": 100.00,
                "items": [
                    {"name": "Item 1", "price": 50.00},
                    {"name": "Item 2", "price": 50.00}
                ]
            }
        """

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {"url": image_base64}
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }
        ]

        response = self.client.chat_completions_create(
            model=VISION_MODEL,
            messages=messages,
            max_tokens=1000,
            response_format={"type": "json_object"}
        )

        content = response["choices"][0]["message"]["content"]

        try:
            receipt_data = json.loads(content)
            if not isinstance(receipt_data, dict):
                return RECEIPT_JSON_TEMPLATE.copy()
            return self.handle_missing_fields(receipt_data)
        except json.JSONDecodeError:
            return RECEIPT_JSON_TEMPLATE.copy()

    def handle_missing_fields(self, receipt_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Handle missing fields in extracted receipt data.

        Normalizes the shape of the data returned by the model so downstream
        code can safely rely on the expected keys/types, even if the model
        omits fields, returns the wrong type, or returns malformed items.
        """
        if not isinstance(receipt_data, dict):
            return RECEIPT_JSON_TEMPLATE.copy()

        for key, default_value in RECEIPT_JSON_TEMPLATE.items():
            if key not in receipt_data or receipt_data[key] is None:
                receipt_data[key] = default_value if not isinstance(default_value, (list, dict)) else type(default_value)()

        # Merchant: must be a string.
        if not isinstance(receipt_data.get("merchant"), str):
            receipt_data["merchant"] = str(receipt_data.get("merchant") or "")

        # Date: must be a string.
        if not isinstance(receipt_data.get("date"), str):
            receipt_data["date"] = str(receipt_data.get("date") or "")

        # Total: must be a valid non-negative number.
        try:
            total = float(receipt_data.get("total", 0.0))
            if total < 0:
                total = 0.0
        except (TypeError, ValueError):
            total = 0.0
        receipt_data["total"] = total

        # Items: must be a list of {"name": str, "price": float} dicts.
        raw_items = receipt_data.get("items")
        if not isinstance(raw_items, list):
            raw_items = []

        normalized_items = []
        for item in raw_items:
            if isinstance(item, dict):
                name = item.get("name")
                price = item.get("price")
            elif isinstance(item, str):
                name = item
                price = None
            else:
                name = None
                price = None

            if not isinstance(name, str) or not name.strip():
                name = ""

            try:
                price = float(price) if price is not None else 0.0
                if price < 0:
                    price = 0.0
            except (TypeError, ValueError):
                price = 0.0

            normalized_items.append({"name": name, "price": price})

        receipt_data["items"] = normalized_items

        return receipt_data

In [ ]:
import time
import hashlib
from typing import Dict, Any, Optional, Callable
from api_client import get_openai_client
from config import MAX_RETRIES, CACHE_TTL_SECONDS


class ProductionHandler:
    """Handles production concerns for receipt processing."""

    def __init__(self):
        """Initialize the ProductionHandler."""
        self.cache: Dict[str, Dict[str, Any]] = {}
        self.client = get_openai_client()

    def retry_with_backoff(
        self,
        func: Callable,
        *args,
        **kwargs
    ) -> Any:
        """
        Execute function with retry logic for rate limit errors.
        """
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if self.is_retryable_error(e):
                for i in range(MAX_RETRIES):
                    delay = 2 ** i
                    time.sleep(delay)
                    try:
                        return func(*args, **kwargs)
                    except Exception as e:
                        if not self.is_retryable_error(e):
                            break
            raise

    def get_cache_key(self, image_base64: str, prompt: str) -> str:
        """
        Generate cache key from image and prompt.
        """
        image_hash = hashlib.md5(image_base64.encode()).hexdigest()
        prompt_hash = hashlib.md5(prompt.encode()).hexdigest()

        cache_key = f"cache:{image_hash}:{prompt_hash}"
        return cache_key

    def is_retryable_error(self, error: Exception) -> bool:
        """
        Classify error as retryable or non-retryable.

        Retryable: 429 (Rate Limit), 500-599 (Server Errors)
        Non-retryable: 400, 401, 403, 404
        """
        if hasattr(error, "status_code"):
            if error.status_code == 429:
                return True
            elif 500 <= error.status_code < 600:
                return True
        return False

    def get_cached_result(self, cache_key: str) -> Optional[Dict[str, Any]]:
        """Get cached result if available and not expired."""
        if cache_key in self.cache:
            entry = self.cache[cache_key]
            if time.time() - entry["timestamp"] < CACHE_TTL_SECONDS:
                return entry["data"]
            else:
                del self.cache[cache_key]
        return None

    def set_cached_result(self, cache_key: str, data: Dict[str, Any]):
        """Store result in cache."""
        self.cache[cache_key] = {
            "data": data,
            "timestamp": time.time()
        }
